# 07 — Comparison and ablations (Stage F)

One comparison table for the three arms (classical GB @ 224, CNN, hybrid; classical GB @ 512 as
the practical-deployment configuration), bootstrap intervals on val, and three ablations, each
changing one thing from the Stage D CNN:

1. training flaw labels at >=1 vote instead of >=2 (classical GB and CNN), scored at >=2;
2. random initialisation instead of ImageNet weights (CNN);
3. frozen backbone instead of full fine-tuning (CNN, Stage D runs).

The backbone comparison (EfficientNet-B0 vs ResNet-50) is not run. Apart from the classical GB
refit at >=1 vote (seconds), this notebook trains nothing; it reads saved val scores and training
histories and regenerates `reports/results_comparison.md` and `reports/results_comparison.csv`.

**Partitions:** every number is on the val partition. The test partition is never read.

**How to run:** from the `notebooks/` folder, with `DATA_DIR` pointing at the data directory
(defaults to `../data`). `DATA_DIR` must contain `annotations/{train,val}.json`,
`classical_features_224.csv`, the Stage C to E prediction files in `predictions/`,
`predictions/cnn_finetune_{votes1,scratch}_val.csv` and the matching `cnn_runs/*/history.json`
(from `scripts/train_cnn_ablations.py`). No GPU needed; about a minute, mostly the bootstrap.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
sys.path.insert(0, str(REPO_DIR))
DATA_DIR = Path(os.environ.get("DATA_DIR", REPO_DIR / "data"))
REPORT_PATH = REPO_DIR / "reports" / "results_comparison.md"
RESULTS_CSV = REPO_DIR / "reports" / "results_comparison.csv"
PREDICTIONS_DIR = DATA_DIR / "predictions"
RUNS_DIR = DATA_DIR / "cnn_runs"

from src.data.splits import MODELLED_FLAWS, SEED, get_splits
from src.eda import report as rp
from src.models import classical as cm
from src.models import metrics as mt

N_RESAMPLES = 1000   # bootstrap resamples of the val images, seed 42
SEEDS = (42, 43, 44)
CLASSICAL, CLASSICAL_512 = "Classical GB @ 224", "Classical GB @ 512"
CNN, HYBRID, FROZEN = "CNN fine-tuned", "Hybrid", "CNN frozen"
CNN_VOTES1, CNN_SCRATCH, CLASSICAL_VOTES1 = "CNN, flaws >= 1 vote", "CNN, random init", "Classical GB @ 224, flaws >= 1 vote"
PREDICTION_FILES = {
    CLASSICAL: "classical_gb_224_val.csv", CLASSICAL_512: "classical_gb_512_val.csv",
    FROZEN: "cnn_frozen_val.csv", CNN_VOTES1: "cnn_finetune_votes1_val.csv",
    CNN_SCRATCH: "cnn_finetune_scratch_val.csv",
    **{(arm, s): f"{pred}_finetune{'' if s == 42 else f'_seed{s}'}_val.csv"
       for arm, pred in ((CNN, "cnn"), (HYBRID, "hybrid")) for s in SEEDS},
}
RUN_DIRS = {FROZEN: "frozen", CNN: "finetune", CNN_VOTES1: "finetune_votes1", CNN_SCRATCH: "finetune_scratch"}
FLAW_TYPE = {"BLR": "photometric", "DRK": "photometric", "BRT": "photometric",
             "FRM": "semantic", "OBS": "semantic", "ROT": "semantic", cm.UNREC: "recognisability"}
ROWS = cm.OUTPUTS + [mt.MACRO]
ENVIRONMENT = rp.environment_lines(
    gpu_note="not needed here: this notebook scores saved predictions and refits one gradient-boosting "
             "baseline; training GPUs are recorded in the Stage D/E reports and below",
    seed_note=f"bootstrap ({N_RESAMPLES} resamples) and gradient boosting use seed 42; CNN entries seed 42")
pd.set_option("display.precision", 3)

print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)
print("\n".join(ENVIRONMENT))

## 1. Data
Val targets at >=2 votes from `src/data/splits.py`. Every prediction file must hold exactly the
val images, one score column per output.

In [ ]:
splits = get_splits(DATA_DIR)
targets = cm.build_targets(splits["val"])


def load_scores(filename):
    s = pd.read_csv(PREDICTIONS_DIR / filename, index_col=0)  # first column: image name
    s.index.name = "image"
    assert sorted(s.index) == sorted(targets.index), f"{filename}: image set differs from val"
    assert list(s.columns) == cm.OUTPUTS, f"{filename}: columns {list(s.columns)}"
    print(f"{filename:<34} {len(s):,} images")
    return s.loc[targets.index]


scores = {name: load_scores(f) for name, f in PREDICTION_FILES.items()}
scores[CNN], scores[HYBRID] = scores[CNN, 42], scores[HYBRID, 42]   # arm entries
sections = []  # (heading, markdown) pairs collected for the report


def evaluate_with_gain(s):
    table = mt.evaluate(targets, s)
    table.insert(table.columns.get_loc("AP") + 1, "gain", mt.normalised_gain(table["AP"], table["base rate"]))
    table.loc[mt.MACRO, "gain"] = np.nan  # gain is read per output, not on the averaged row
    return table


results = {name: evaluate_with_gain(s) for name, s in scores.items()}
base_rate = results[CNN].loc[ROWS, "base rate"]

## 2. The comparison table
Flaws down the side, arms across. CNN and hybrid: the seed-42 entry plus the mean and range over
seeds 42-44 (run-to-run spread). Classical GB @ 512 is the practical-deployment configuration,
not one of the matched-input arms.

In [ ]:
def seed_stats(arm):
    aps = pd.concat([results[arm, s].loc[ROWS, "AP"] for s in SEEDS], axis=1).astype(float)
    return aps.mean(axis=1), aps.min(axis=1), aps.max(axis=1)


comparison = pd.DataFrame({"type": [FLAW_TYPE.get(o, "") for o in ROWS], "base rate": base_rate}, index=ROWS)
comparison[CLASSICAL] = results[CLASSICAL].loc[ROWS, "AP"]
for arm in (CNN, HYBRID):
    mean, low, high = seed_stats(arm)
    comparison[f"{arm} (seed 42)"] = results[arm].loc[ROWS, "AP"]
    comparison[f"{arm} mean of 3 seeds"] = mean
    comparison[f"{arm} seed range"] = [f"{a:.3f}-{b:.3f}" for a, b in zip(low, high)]
comparison[f"{CLASSICAL_512} (deployment)"] = results[CLASSICAL_512].loc[ROWS, "AP"]
display(comparison)

gains = pd.DataFrame({name: results[name].loc[cm.OUTPUTS, "gain"] for name in (CLASSICAL, CNN, HYBRID, CLASSICAL_512)})
by_type = gains.loc[MODELLED_FLAWS].groupby(pd.Series(FLAW_TYPE)).mean()
f1 = pd.DataFrame({name: results[name].loc[ROWS, "F1"] for name in (CLASSICAL, CNN, HYBRID, CLASSICAL_512)})
print("Gain by output"); display(gains)
print("Mean gain by flaw type"); display(by_type)
print("F1 @ 0.5"); display(f1)

sections.append(("Comparison table (val AP)", (
    "Same val partition (3,875 images, >=2-vote labels) and the same metrics code in every cell. CNN "
    "and hybrid: seed-42 entry, mean and range over seeds 42-44.\n\n" + rp.df_to_markdown(comparison.round(3))
    + "\n\nGain = (AP - base rate) / (1 - base rate):\n\n" + rp.df_to_markdown(gains.round(3))
    + "\n\nMean gain by flaw type:\n\n" + rp.df_to_markdown(by_type.round(3))
    + "\n\nF1 at threshold 0.5:\n\n" + rp.df_to_markdown(f1.round(3)))))

## 3. The pre-registered 224 vs 512 rule
The rule (docs/decisions.md, 2026-10-04) anticipated 224 = 512 on BLR (resolution not the driver)
or 512 better (blur needs detail 224 px removes). Stage C found 224 clearly better, outside the
rule; it was recorded as a finding and the rule was not amended. Restated here on the final table.

In [ ]:
rule = pd.DataFrame({CLASSICAL: results[CLASSICAL].loc[["BLR", mt.MACRO], "AP"],
                     CLASSICAL_512: results[CLASSICAL_512].loc[["BLR", mt.MACRO], "AP"]})
rule["224 - 512"] = rule[CLASSICAL] - rule[CLASSICAL_512]
display(rule)
rule_diff = mt.paired_difference(targets, scores[CLASSICAL], scores[CLASSICAL_512], N_RESAMPLES)
display(rule_diff.loc[["BLR", mt.MACRO]])
sections.append(("Pre-registered 224 vs 512 rule", (
    "Classical GB, BLR and macro AP, with the paired bootstrap interval of the 224 - 512 difference:"
    "\n\n" + rp.df_to_markdown(rule.round(3)) + "\n\n"
    + rp.df_to_markdown(rule_diff.loc[["BLR", mt.MACRO]].round(3)))))

## 4. Bootstrap intervals
95% percentile intervals from 1,000 resamples of the val images (seed 42), with every model scored
on the same resampled images, so differences are paired. They show how much each AP depends on
which images happen to be in the val partition; training randomness is the separate seed spread.
An interval of a difference that excludes 0 means the gap is larger than that sampling variation.

In [ ]:
intervals = mt.ap_with_interval(targets, {n: scores[n] for n in (CLASSICAL, CNN, HYBRID)}, N_RESAMPLES)
ci = pd.DataFrame({n: [f"{r.AP:.3f} [{r.low:.3f}, {r.high:.3f}]" for r in t.itertuples()]
                   for n, t in intervals.items()}, index=intervals[CNN].index)
print("AP [95% interval]"); display(ci)

pairs = {"CNN - classical": (CNN, CLASSICAL), "hybrid - CNN": (HYBRID, CNN)}
differences = {label: mt.paired_difference(targets, scores[a], scores[b], N_RESAMPLES) for label, (a, b) in pairs.items()}
for label, table in differences.items():
    print(label); display(table)
sections.append(("Bootstrap intervals (val)", (
    "AP with 95% bootstrap interval (1,000 resamples of the val images, seed 42):\n\n" + rp.df_to_markdown(ci)
    + "".join(f"\n\nPaired difference, {label}:\n\n" + rp.df_to_markdown(t.round(3))
              for label, t in differences.items()))))

## 5. Ablation 1: training labels at >=1 vote
At >=1 vote the rare flaws gain many positives (OBS 840 -> 3,268 on train) at the cost of noisier
labels. Training flaw labels are switched to >=1 vote; UNREC stays at >=2, and scoring (and, for
the CNN, early stopping) stays at >=2, so the task measured is unchanged. The classical GB is refit
here (seconds); the CNN run comes from `scripts/train_cnn_ablations.py`.

In [ ]:
train_records = splits["train"]
train_counts = pd.DataFrame({">= 2 votes": cm.build_targets(train_records).sum(),
                             ">= 1 vote": cm.build_targets(train_records, flaw_threshold=1).sum()})
train_counts["ratio"] = train_counts[">= 1 vote"] / train_counts[">= 2 votes"]
print("Training positives"); display(train_counts)

features = cm.load_features(DATA_DIR, 224)
train_targets_votes1 = cm.build_targets(train_records, flaw_threshold=1)
models_votes1 = cm.train_models(features.loc[train_targets_votes1.index], train_targets_votes1, "gb")
scores[CLASSICAL_VOTES1] = cm.predict_scores(models_votes1, features.loc[targets.index])
scores[CLASSICAL_VOTES1].to_csv(PREDICTIONS_DIR / "classical_gb_224_votes1_val.csv")
results[CLASSICAL_VOTES1] = evaluate_with_gain(scores[CLASSICAL_VOTES1])

votes = pd.DataFrame({"base rate": base_rate,
                      f"{CLASSICAL} (>= 2)": results[CLASSICAL].loc[ROWS, "AP"],
                      CLASSICAL_VOTES1: results[CLASSICAL_VOTES1].loc[ROWS, "AP"],
                      f"{CNN} (>= 2)": results[CNN].loc[ROWS, "AP"],
                      CNN_VOTES1: results[CNN_VOTES1].loc[ROWS, "AP"]})
print("Val AP, scored at >= 2 votes"); display(votes)
votes_diff = {"classical: >= 1 - >= 2": mt.paired_difference(targets, scores[CLASSICAL_VOTES1], scores[CLASSICAL], N_RESAMPLES),
              "CNN: >= 1 - >= 2": mt.paired_difference(targets, scores[CNN_VOTES1], scores[CNN], N_RESAMPLES)}
for label, table in votes_diff.items():
    print(label); display(table)
sections.append(("Ablation 1: training labels at >=1 vote (val, scored at >=2)", (
    "Training positives:\n\n" + rp.df_to_markdown(train_counts.round(2)) + "\n\nVal AP:\n\n"
    + rp.df_to_markdown(votes.round(3))
    + "".join(f"\n\nPaired difference, {label}:\n\n" + rp.df_to_markdown(t.round(3))
              for label, t in votes_diff.items()))))

## 6. Ablation 2: ImageNet weights vs random initialisation
The same network and recipe, starting from random weights. The recipe (learning rates, at most 15
epochs) was set for a pretrained network; a from-scratch network would do somewhat better with a
longer, tuned schedule, so the gap is an upper bound on what pretraining contributes under this
recipe.

In [ ]:
def ablation_table(a, b):
    table = pd.DataFrame({"type": [FLAW_TYPE.get(o, "") for o in ROWS], "base rate": base_rate,
                          f"AP {a}": results[a].loc[ROWS, "AP"], f"AP {b}": results[b].loc[ROWS, "AP"],
                          f"gain {a}": results[a].loc[ROWS, "gain"], f"gain {b}": results[b].loc[ROWS, "gain"]})
    by_type = table.loc[MODELLED_FLAWS].groupby("type")[[f"gain {a}", f"gain {b}"]].mean()
    return table, by_type


def run_summary(names):
    rows = {}
    for name in names:
        h = json.loads((RUNS_DIR / RUN_DIRS[name] / "history.json").read_text())
        e = pd.DataFrame(h["epochs"])
        rows[name] = {"epochs run": len(e), "best epoch": h["best_epoch"],
                      "best val macro AP": round(h["best_val_macro_ap"], 4),
                      "train loss at best epoch": round(e.set_index("epoch").loc[h["best_epoch"], "train_loss"], 3),
                      "torch": h["environment"]["libraries"].split("torch ")[1].split(",")[0]}
    return pd.DataFrame(rows).T


scratch, scratch_by_type = ablation_table(CNN, CNN_SCRATCH)
scratch_runs = run_summary((CNN, CNN_SCRATCH))
scratch_diff = mt.paired_difference(targets, scores[CNN], scores[CNN_SCRATCH], N_RESAMPLES)
display(scratch_runs); display(scratch); display(scratch_by_type)
print("pretrained - random init"); display(scratch_diff)
sections.append(("Ablation 2: ImageNet weights vs random initialisation (val)", (
    rp.df_to_markdown(scratch_runs) + "\n\n" + rp.df_to_markdown(scratch.round(3))
    + "\n\nMean gain by flaw type:\n\n" + rp.df_to_markdown(scratch_by_type.round(3))
    + "\n\nPaired difference, pretrained - random init:\n\n" + rp.df_to_markdown(scratch_diff.round(3)))))

## 7. Ablation 3: frozen backbone vs full fine-tuning
The Stage D runs: the frozen backbone (heads only, 5 epochs) against full fine-tuning. No new
training.

In [ ]:
frozen, frozen_by_type = ablation_table(CNN, FROZEN)
frozen_runs = run_summary((CNN, FROZEN))
frozen_diff = mt.paired_difference(targets, scores[CNN], scores[FROZEN], N_RESAMPLES)
display(frozen_runs); display(frozen); display(frozen_by_type)
print("fine-tuned - frozen"); display(frozen_diff)
sections.append(("Ablation 3: frozen backbone vs full fine-tuning (val)", (
    rp.df_to_markdown(frozen_runs) + "\n\n" + rp.df_to_markdown(frozen.round(3))
    + "\n\nMean gain by flaw type:\n\n" + rp.df_to_markdown(frozen_by_type.round(3))
    + "\n\nPaired difference, fine-tuned - frozen:\n\n" + rp.df_to_markdown(frozen_diff.round(3)))))

## 8. Ablation 4: backbone comparison (not run)
EfficientNet-B0 vs ResNet-50 is the lowest-priority ablation and the first to drop under the
compute contingency; it was not run.

In [ ]:
sections.append(("Ablation 4: backbone comparison", (
    "Not run. EfficientNet-B0 vs ResNet-50 is the lowest-priority ablation in the plan and the first to "
    "drop under the compute and time contingency.")))

## 9. Write the report
The findings block in `reports/results_comparison.md` is hand-written after reading the results
and is preserved on every rerun.

In [ ]:
long = pd.concat({(n if isinstance(n, str) else f"{n[0]}, seed {n[1]}"): t for n, t in results.items()
                  if not (isinstance(n, str) and n in (CNN, HYBRID))}, names=["model", "output"])
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
long.reset_index().to_csv(RESULTS_CSV, index=False, float_format="%.4f")
out = rp.write_report(REPORT_PATH, sections, title="Stage F: Comparison and ablations",
                      environment=ENVIRONMENT, findings_heading="Findings")
print("Report written to", out)
print("Results table written to", RESULTS_CSV)